# ⚡ Kronumos 70B/72B Kairos — Fast Inference & Rapid Benchmark on NVIDIA A100
**Cognitive Neural Cortex:** `unsloth/Qwen2.5-72B-Instruct-bnb-4bit` (NF4 4-bit Pre-quantized, ~39 GB download, ~35 GB VRAM)
**Deterministic Sub-Cortex:** Tokenectomy Native Rust Engine (`libtokenectomy_subcortex.so`, 5µs `IndentationHealer`, `ScopeGuard`)
**Hardware:** NVIDIA A100 (40GB or 80GB SXM4) | Precision: `bfloat16` compute with 4-bit weights

Notebook ini dirancang khusus untuk **uji inferensi cepat (rapid intervention)** model kelas 70B/72B yang dipasangkan langsung dengan **Rust Sub-Cortex C-ABI Engine** kita tanpa dirty diff.

### Step 1: Verifikasi GPU A100 & VRAM
Pastikan runtime menggunakan A100 GPU (Runtime -> Change runtime type -> A100 GPU).

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ GPU CUDA wajib aktif! Ubah runtime ke A100 di menu Runtime -> Change runtime type."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ Active Accelerator: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"✅ bfloat16 Hardware Acceleration: {torch.cuda.is_bf16_supported()}")

### Step 2: Clone Repo, Install Stack, & Load Native Rust Sub-Cortex

In [ ]:
# 1. Clone repository resmi Kronomus
import os
if not os.path.exists("Kronomus"):
    !git clone https://github.com/Tokenectomy-Labs/Kronomus.git
    %cd Kronomus
else:
    %cd Kronomus
    !git pull

# 2. Install library inferensi terbaru (BitsAndBytes, Accelerate, Transformers, Datasets)
!pip install -q -U transformers accelerate bitsandbytes datasets peft

# 3. Unduh / Load Native Rust Sub-Cortex Engine
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None

if not gh_token:
    gh_token = os.environ.get('GITHUB_TOKEN', '')

if not gh_token:
    from getpass import getpass
    gh_token = getpass('🔑 Masukkan GitHub Token (untuk akses Rust engine binary & API limits): ')
    os.environ['GITHUB_TOKEN'] = gh_token

if not os.path.exists("libtokenectomy_subcortex.so"):
    !curl -s -H "Authorization: token {gh_token}" \
          -L "https://raw.githubusercontent.com/Tokenectomy-Labs/Tokenectomy-Sentinel/main/bin/libtokenectomy_subcortex.so" \
          -o libtokenectomy_subcortex.so

# 4. Verifikasi Sub-Cortex Engine Berhasil Aktif
!python3 -c "from scripts.tokenectomy_subcortex_rust import RustSubCortex; print('🎉 Rust Sub-Cortex Status:', '100% NATIVE C-ABI AKTIF' if RustSubCortex.is_available() else 'Python Fallback Mode')"

### Step 3: Interactive Rapid Single-Issue Test (72B Model + Sub-Cortex)
Jalankan cell ini untuk menguji inferensi 1 soal riil dari SWE-bench Verified (misal: `astropy__astropy-12907`).
Objek `runner` akan tersimpan di memory VRAM GPU dan siap digunakan langsung untuk batch run di Step 4 tanpa reload ulang.

In [ ]:
import time, json, torch
from datasets import load_dataset
from scripts.kaggle_kronumos_runner import KronumosBenchmarkRunner

# Inisialisasi model 72B pre-quantized ke VRAM A100
MODEL_ID = "unsloth/Qwen2.5-72B-Instruct-bnb-4bit"

if 'runner' not in globals():
    print(f"🚀 Menginisialisasi Kronumos 72B Runner dengan {MODEL_ID}...")
    runner = KronumosBenchmarkRunner(model_id=MODEL_ID, load_in_4bit=True, github_token=gh_token)
else:
    print("⚡ Model 72B sudah siap di VRAM GPU!")

# Ambil 1 sampel riil SWE-bench Verified
print("📥 Mengambil dataset princeton-nlp/SWE-bench_Verified...")
ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
sample_inst = ds[0] # astropy__astropy-12907

print(f"\n🔥 MEMULAI RAPID INTERVENTION PADA: {sample_inst['instance_id']}")
print(f"Repository: {sample_inst['repo']}")
t0 = time.time()
res = runner.solve_instance(sample_inst, max_turns=3)
elapsed = round(time.time() - t0, 2)

print("\n" + "=" * 60)
print(f"⏱️ Waktu Inferensi: {elapsed} detik")
print(f"📊 Total Tokens: {res['total_tokens']} (Prompt: {res['prompt_tokens']} | Completion: {res['completion_tokens']})")
print(f"✅ Patch Berhasil Dibuat: {'YA' if res['model_patch'] else 'TIDAK'}")
print("=" * 60)
if res['model_patch']:
    print("\n📄 HASIL UNIFIED DIFF YANG SUDAH DI-HEAL OLEH SUB-CORTEX:")
    print(res['model_patch'])
else:
    print("❌ Gagal menghasilkan patch valid pada turn ini.")

### Step 4: Rapid Batch Benchmark Run (5 Sampel Langsung dari VRAM)
Cell ini **memanfaatkan model 72B yang SUDAH ADA di VRAM GPU** dari Step 3, sehingga **TIDAK PERLU RELOAD ULANG**, bebas dari bentrok alokasi VRAM, dan langsung mengeksekusi 5 soal berturut-turut!

In [ ]:
import os, json, time, torch

output_dir = "output_colab_72b_5"
os.makedirs(output_dir, exist_ok=True)
pred_file = os.path.join(output_dir, "predictions.jsonl")
metrics_file = os.path.join(output_dir, "eval_metrics.json")

num_eval = 5
instances = [ds[i] for i in range(num_eval)]
results = []

print(f"🎯 Menjalankan batch {num_eval} soal SWE-bench Verified langsung dari VRAM A100...\n")

for i, inst in enumerate(instances):
    iid = inst["instance_id"]
    repo = inst.get("repo", "unknown")
    print(f"[{i+1}/{num_eval}] 🔧 Memproses: {iid} ({repo})...")
    
    t0 = time.time()
    res = runner.solve_instance(inst, max_turns=3)
    elapsed = round(time.time() - t0, 2)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    pred_entry = {
        "instance_id": iid,
        "model_patch": res["model_patch"],
        "model_name_or_path": "Qwen2.5-72B-Instruct"
    }
    
    with open(pred_file, "a") as pf:
        pf.write(json.dumps(pred_entry) + "\n")
        
    has_patch = bool(res["model_patch"])
    results.append({
        "instance_id": iid,
        "has_patch": has_patch,
        "total_tokens": res["total_tokens"],
        "latency_sec": elapsed
    })
    
    status_badge = "✅ PATCH VALID" if has_patch else "❌ TANPA PATCH"
    print(f"    ↳ {status_badge} | Tokens: {res['total_tokens']} | Waktu: {elapsed}s\n")

with open(metrics_file, "w") as mf:
    json.dump({"summary": results}, mf, indent=2)

valid_count = sum(1 for r in results if r["has_patch"])
print("=" * 60)
print(f"🎉 BATCH RUN SELESAI! {valid_count}/{num_eval} soal menghasilkan patch valid!")
print(f"📁 File Prediksi: {pred_file}")
print("=" * 60)

### Step 5: Download Hasil Prediksi (predictions.jsonl)
Unduh hasil prediksi yang sudah disintesis oleh 72B/70B + Sub-Cortex untuk evaluasi Docker.

In [ ]:
from google.colab import files
import os

if os.path.exists("output_colab_72b_5/predictions.jsonl"):
    !zip -r kronumos_72b_predictions.zip output_colab_72b_5/predictions.jsonl output_colab_72b_5/eval_metrics.json
    files.download('kronumos_72b_predictions.zip')
else:
    print("⚠️ File predictions.jsonl belum ditemukan. Jalankan Step 4 terlebih dahulu!")